# Visualization 1 - Baby Names Through Time

This notebook creates **one simple static visualization** for the first mini-project question:

- Which names stay popular for a long time?
- Which names become popular only briefly?
- How does popularity change from 1900 to 2020?
- Do French baby names move in waves over time?

This notebook uses **small multiples** built from a small set of **signature names** drawn from different periods of the dataset.

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [2]:
# Load and clean the department-level dataset, then aggregate it to national level.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)

national = names.groupby(['annais', 'preusuel'], as_index=False)['nombre'].sum()

# Keep only the years 2000-2020, then compute ranks for a broader top set.
rank_data = national.copy()
rank_data = rank_data[(rank_data['annais'] >= 2000) & (rank_data['annais'] <= 2020)].copy()
rank_data['rank'] = rank_data.groupby('annais')['nombre'].rank(method='first', ascending=False)
rank_data = rank_data[rank_data['rank'] <= 50].copy()

# Keep integer ranks for a flexible y-axis.
rank_data['rank'] = rank_data['rank'].astype(int)
rank_data = rank_data.sort_values(['preusuel', 'annais']).copy()

# Break lines when a name leaves the selected ranking band for one or more years.
rank_data['prev_year'] = rank_data.groupby('preusuel')['annais'].shift()
rank_data['new_segment'] = ((rank_data['prev_year'].isna()) | ((rank_data['annais'] - rank_data['prev_year']) > 1)).astype(int)
rank_data['segment'] = rank_data.groupby('preusuel')['new_segment'].cumsum()
rank_data['line_group'] = rank_data['preusuel'] + '_' + rank_data['segment'].astype(str)

rank_data.head()

,annais,preusuel,nombre,rank,prev_year,new_segment,segment,line_group
227276,2016,AARON,2363,40,NaN,1,1,AARON_1
231730,2017,AARON,2372,35,2016.0,0,1,AARON_1
236142,2018,AARON,2239,36,2017.0,0,1,AARON_1
240536,2019,AARON,2436,34,2018.0,0,1,AARON_1
244878,2020,AARON,2310,32,2019.0,0,1,AARON_1


In [3]:
max_rank = alt.param(name='max_rank', value=10, bind=alt.binding_range(min=5, max=50, step=1, name='Top ranks shown '))
year_start = alt.param(name='year_start', value=2000, bind=alt.binding_range(min=2000, max=2020, step=1, name='From year '))
year_end = alt.param(name='year_end', value=2020, bind=alt.binding_range(min=2000, max=2020, step=1, name='To year '))
name_hover = alt.selection_point(fields=['preusuel'], on='mouseover', nearest=True, empty='none')

base = alt.Chart(rank_data).transform_filter(
    (alt.datum.rank <= max_rank)
    & (alt.datum.annais >= year_start)
    & (alt.datum.annais <= year_end)
)

lines = base.mark_line(strokeWidth=1.8).encode(
    x=alt.X('annais:Q', title='Year', scale=alt.Scale(zero=False, nice=False), axis=alt.Axis(format='d', labelAngle=0)),
    y=alt.Y('rank:Q', title='Rank within the year', scale=alt.Scale(domainMin=1, domainMax=max_rank, zero=False, reverse=True, nice=False), axis=alt.Axis(tickMinStep=1)),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    detail='line_group:N',
    order=alt.Order('annais:Q'),
    opacity=alt.condition(name_hover, alt.value(1), alt.value(0.45)),
    tooltip=[
        alt.Tooltip('annais:Q', title='Year'),
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('rank:Q', title='Rank'),
        alt.Tooltip('nombre:Q', title='Births')
    ]
)

points = base.mark_circle(size=40).encode(
    x=alt.X('annais:Q', scale=alt.Scale(zero=False, nice=False)),
    y=alt.Y('rank:Q', scale=alt.Scale(domainMin=1, domainMax=max_rank, zero=False, reverse=True, nice=False)),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    opacity=alt.condition(name_hover, alt.value(1), alt.value(0.7)),
    tooltip=[
        alt.Tooltip('annais:Q', title='Year'),
        alt.Tooltip('preusuel:N', title='Name'),
        alt.Tooltip('rank:Q', title='Rank'),
        alt.Tooltip('nombre:Q', title='Births')
    ]
)

labels = base.transform_window(
    first_visible_point='row_number()',
    sort=[alt.SortField('annais', order='ascending')],
    groupby=['preusuel']
).transform_filter(
    alt.datum.first_visible_point == 1
).mark_text(fontSize=8, dy=-8, dx=6, align='left').encode(
    x=alt.X('annais:Q', scale=alt.Scale(zero=False, nice=False)),
    y=alt.Y('rank:Q', scale=alt.Scale(domainMin=1, domainMax=max_rank, zero=False, reverse=True, nice=False)),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    text='preusuel:N',
    opacity=alt.condition(name_hover, alt.value(1), alt.value(0.5))
)

(lines + points + labels).add_params(
    max_rank,
    year_start,
    year_end,
    name_hover
).properties(
    width=1000,
    height=360,
    title='Interactive Ranking of Baby Names in France (2000-2020)'
)

alt.LayerChart(...)

## Why this visualization fits the assignment

### Advantages

- The yearly top 10 ranking makes comparison across years immediate.
- The connected lines help reveal names that stay stable, rise gradually, or suddenly appear.
- The text labels on each point make the ranking readable without relying only on color.

### Disadvantages

- It only shows the top 10 names, so less popular names and long-tail patterns are hidden, it may miss interesting trends outside the top 10.
- Can add more than 10 names, but it would harden readability and make the chart more cluttered.
- With many labels and lines, some years can still feel visually busy.
- It explains rank well, but not the absolute number of births behind each name.